# Get those with AFIB code

In [1]:
library(tidyverse)
library(bigrquery)

# This query represents dataset "afib" for domain "condition" and was generated for All of Us Controlled Tier Dataset v7
dataset_93775608_condition_sql <- paste("
    SELECT
        c_occurrence.person_id,
        c_occurrence.condition_concept_id,
        c_standard_concept.concept_name as standard_concept_name,
        c_standard_concept.concept_code as standard_concept_code,
        c_standard_concept.vocabulary_id as standard_vocabulary,
        c_occurrence.condition_start_datetime 
    FROM
        ( SELECT
            * 
        FROM
            `condition_occurrence` c_occurrence 
        WHERE
            (
                condition_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `cb_criteria` cr       
                    WHERE
                        concept_id IN (313217)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 1 
                    AND is_selectable = 1)
            )  
            AND (
                c_occurrence.PERSON_ID IN (SELECT
                    distinct person_id  
                FROM
                    `cb_search_person` cb_search_person  
                WHERE
                    cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `cb_search_person` p 
                    WHERE
                        has_whole_genome_variant = 1 ) 
                    AND cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `cb_search_person` p 
                    WHERE
                        has_ehr_data = 1 ) )
            )) c_occurrence 
    LEFT JOIN
        `concept` c_standard_concept 
            ON c_occurrence.condition_concept_id = c_standard_concept.concept_id", sep="")

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
condition_93775608_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "condition_93775608",
  "condition_93775608_*.csv")
message(str_glue('The data will be written to {condition_93775608_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), dataset_93775608_condition_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  condition_93775608_path,
  destination_format = "CSV")


# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {condition_93775608_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(standard_concept_name = col_character(), standard_concept_code = col_character(), standard_vocabulary = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
condition_df <- read_bq_export_from_workspace_bucket(condition_93775608_path)


── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.2     ✔ tibble    3.2.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.0.4     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
The data will be written to gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/bq_exports/micah_hysong@researchallofus.org/20260423/condition_93775608/condition_93775608_*.csv. Use this path when reading the data into your notebooks in the future.

Loading gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/bq_exports/micah_hysong@researchallofus.org/20260423/condition_93775608/condition_93775608_000000000000.csv.



In [2]:
unique(condition_df$standard_concept_name)

[1] "Paroxysmal atrial fibrillation with rapid ventricular response"
 [2] "Permanent atrial fibrillation"                                 
 [3] "Longstanding persistent atrial fibrillation"                   
 [4] "Rapid atrial fibrillation"                                     
 [5] "Lone atrial fibrillation"                                      
 [6] "Atrial fibrillation with rapid ventricular response"           
 [7] "Atrial fibrillation and flutter"                               
 [8] "Persistent atrial fibrillation"                                
 [9] "Chronic atrial fibrillation"                                   
[10] "Paroxysmal atrial fibrillation"                                
[11] "Atrial fibrillation"                                           
[12] "Controlled atrial fibrillation"

In [3]:
# Count unique condition_start_datetime for each person_id
number_condition_entries <- condition_df %>%
  group_by(person_id) %>%
  summarise(unique_conditions = n_distinct(condition_start_datetime))

In [4]:
# Only those with at least two entries are cases
condition_entries_cases <- number_condition_entries[number_condition_entries$unique_conditions>=2, ]
nrow(condition_entries_cases)

[1] 17283

# Get survey cases

In [5]:
library(tidyverse)
library(bigrquery)

# This query represents dataset "afib_survey" for domain "survey" and was generated for All of Us Controlled Tier Dataset v8
dataset_93211592_survey_sql <- paste("
    SELECT
        answer.person_id,
        answer.survey_datetime,
        answer.survey,
        answer.question_concept_id,
        answer.question,
        answer.answer_concept_id,
        answer.answer,
        answer.survey_version_concept_id,
        answer.survey_version_name  
    FROM
        `ds_survey` answer   
    WHERE
        (
            question_concept_id IN (836828)
        )  
        AND (
            answer.PERSON_ID IN (SELECT
                distinct person_id  
            FROM
                `cb_search_person` cb_search_person  
            WHERE
                cb_search_person.person_id IN (SELECT
                    person_id 
                FROM
                    `cb_search_person` p 
                WHERE
                    has_whole_genome_variant = 1 ) 
                AND cb_search_person.person_id IN (SELECT
                    person_id 
                FROM
                    `cb_search_person` p 
                WHERE
                    has_ehr_data = 1 ) )
        )", sep="")

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
survey_93211592_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "survey_93211592",
  "survey_93211592_*.csv")
message(str_glue('The data will be written to {survey_93211592_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), dataset_93211592_survey_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  survey_93211592_path,
  destination_format = "CSV")


# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {survey_93211592_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(survey = col_character(), question = col_character(), answer = col_character(), survey_version_name = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
survey_df <- read_bq_export_from_workspace_bucket(survey_93211592_path)

The data will be written to gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/bq_exports/micah_hysong@researchallofus.org/20260423/survey_93211592/survey_93211592_*.csv. Use this path when reading the data into your notebooks in the future.

Loading gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/bq_exports/micah_hysong@researchallofus.org/20260423/survey_93211592/survey_93211592_000000000000.csv.



In [6]:
afib<-survey_df[survey_df$answer == "Including yourself, who in your family has had atrial fibrillation (or a-fib) or atrial flutter (or a-flutter)? - Self",]
case_definition2<-unique(afib$person_id)
length(case_definition2)

[1] 9927

# Get all Cases

In [7]:
case_definition1<-unique(condition_entries_cases$person_id)
cases<-unique(c(case_definition1, case_definition2))
length(cases)

[1] 21613

# Get exclusion criteria

In [8]:
# This query represents dataset "afib" for domain "condition" and was generated for All of Us Controlled Tier Dataset v7
dataset_84516414_condition_sql <- paste("
    SELECT
        c_occurrence.person_id,
        c_occurrence.condition_concept_id,
        c_standard_concept.concept_name as standard_concept_name,
        c_standard_concept.concept_code as standard_concept_code,
        c_standard_concept.vocabulary_id as standard_vocabulary 
    FROM
        ( SELECT
            * 
        FROM
            `condition_occurrence` c_occurrence 
        WHERE
            (
                condition_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `cb_criteria` cr       
                    WHERE
                        concept_id IN (42538117, 42538119, 443575)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 1 
                    AND is_selectable = 1)
            )  
            AND (
                c_occurrence.PERSON_ID IN (SELECT
                    distinct person_id  
                FROM
                    `cb_search_person` cb_search_person  
                WHERE
                    cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `cb_search_person` p 
                    WHERE
                        has_whole_genome_variant = 1 ) 
                    AND cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `cb_search_person` p 
                    WHERE
                        has_ehr_data = 1 ) )
            )) c_occurrence 
    LEFT JOIN
        `concept` c_standard_concept 
            ON c_occurrence.condition_concept_id = c_standard_concept.concept_id", sep="")

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
condition_84516414_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "condition_84516414",
  "condition_84516414_*.csv")
message(str_glue('The data will be written to {condition_84516414_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), dataset_84516414_condition_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  condition_84516414_path,
  destination_format = "CSV")


# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {condition_84516414_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(standard_concept_name = col_character(), standard_concept_code = col_character(), standard_vocabulary = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
exclusion_df_1 <- read_bq_export_from_workspace_bucket(condition_84516414_path)

The data will be written to gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/bq_exports/micah_hysong@researchallofus.org/20260423/condition_84516414/condition_84516414_*.csv. Use this path when reading the data into your notebooks in the future.

Loading gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/bq_exports/micah_hysong@researchallofus.org/20260423/condition_84516414/condition_84516414_000000000000.csv.



In [9]:
unique(exclusion_df_1$standard_concept_name)

[1] "Transplanted heart-lung present"                                       
 [2] "Heart-lung transplant failure and rejection"                           
 [3] "Arteriosclerosis of coronary artery bypass graft of transplanted heart"
 [4] "Cardiac transplant failure"                                            
 [5] "Coronary arteriosclerosis in artery of transplanted heart"             
 [6] "Cardiac transplant rejection"                                          
 [7] "Acute rejection of cardiac transplant"                                 
 [8] "Disorder of transplanted heart"                                        
 [9] "Transplanted heart present"                                            
[10] "Transplanted heart valve present"

In [10]:
library(tidyverse)
library(bigrquery)

# This query represents dataset "afib" for domain "procedure" and was generated for All of Us Controlled Tier Dataset v7
dataset_02762000_procedure_sql <- paste("
    SELECT
        procedure.person_id,
        procedure.procedure_concept_id,
        p_standard_concept.concept_name as standard_concept_name,
        p_standard_concept.concept_code as standard_concept_code,
        p_standard_concept.vocabulary_id as standard_vocabulary 
    FROM
        ( SELECT
            * 
        FROM
            `procedure_occurrence` procedure 
        WHERE
            (
                procedure_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `cb_criteria` cr       
                    WHERE
                        concept_id IN (4105593, 4137127)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 1 
                    AND is_selectable = 1)
            )  
            AND (
                procedure.PERSON_ID IN (SELECT
                    distinct person_id  
                FROM
                    `cb_search_person` cb_search_person  
                WHERE
                    cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `cb_search_person` p 
                    WHERE
                        has_whole_genome_variant = 1 ) 
                    AND cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `cb_search_person` p 
                    WHERE
                        has_ehr_data = 1 ) )
            )) procedure 
    LEFT JOIN
        `concept` p_standard_concept 
            ON procedure.procedure_concept_id = p_standard_concept.concept_id", sep="")

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
procedure_02762000_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "procedure_02762000",
  "procedure_02762000_*.csv")
message(str_glue('The data will be written to {procedure_02762000_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), dataset_02762000_procedure_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  procedure_02762000_path,
  destination_format = "CSV")


# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {procedure_02762000_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(standard_concept_name = col_character(), standard_concept_code = col_character(), standard_vocabulary = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
exclusion_df_2 <- read_bq_export_from_workspace_bucket(procedure_02762000_path)

The data will be written to gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/bq_exports/micah_hysong@researchallofus.org/20260423/procedure_02762000/procedure_02762000_*.csv. Use this path when reading the data into your notebooks in the future.

Loading gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/bq_exports/micah_hysong@researchallofus.org/20260423/procedure_02762000/procedure_02762000_000000000000.csv.



In [11]:
unique(exclusion_df_2$standard_concept_name)

[1] "Transplantation of heart valve" "Transplantation of heart"

In [12]:
nrow(exclusion_df_1)
nrow(exclusion_df_2)
exclusion <- unique(c(exclusion_df_1$person_id, exclusion_df_2$person_id))
length(exclusion)
cases<-setdiff(cases, exclusion)
length(cases)

[1] 196845

[1] 905

[1] 4016

[1] 19569

# Get Control Criteria

In [13]:
library(tidyverse)
library(bigrquery)

# This query represents dataset "ecg" for domain "condition" and was generated for All of Us Controlled Tier Dataset v7
dataset_79586544_condition_sql <- paste("
    SELECT
        c_occurrence.person_id,
        c_occurrence.condition_concept_id,
        c_standard_concept.concept_name as standard_concept_name,
        c_standard_concept.concept_code as standard_concept_code,
        c_standard_concept.vocabulary_id as standard_vocabulary 
    FROM
        ( SELECT
            * 
        FROM
            `condition_occurrence` c_occurrence 
        WHERE
            (
                condition_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `cb_criteria` cr       
                    WHERE
                        concept_id IN (4117134)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 1 
                    AND is_selectable = 1)
            )  
            AND (
                c_occurrence.PERSON_ID IN (SELECT
                    distinct person_id  
                FROM
                    `cb_search_person` cb_search_person  
                WHERE
                    cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `cb_search_person` p 
                    WHERE
                        has_whole_genome_variant = 1 ) 
                    AND cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `cb_search_person` p 
                    WHERE
                        has_ehr_data = 1 ) )
            )) c_occurrence 
    LEFT JOIN
        `concept` c_standard_concept 
            ON c_occurrence.condition_concept_id = c_standard_concept.concept_id", sep="")

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
condition_79586544_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "condition_79586544",
  "condition_79586544_*.csv")
message(str_glue('The data will be written to {condition_79586544_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), dataset_79586544_condition_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  condition_79586544_path,
  destination_format = "CSV")


# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {condition_79586544_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(standard_concept_name = col_character(), standard_concept_code = col_character(), standard_vocabulary = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
ecg_df_1 <- read_bq_export_from_workspace_bucket(condition_79586544_path)

library(tidyverse)
library(bigrquery)

# This query represents dataset "ecg" for domain "measurement" and was generated for All of Us Controlled Tier Dataset v7
dataset_79586544_measurement_sql <- paste("
    SELECT
        measurement.person_id,
        measurement.measurement_concept_id,
        m_standard_concept.concept_name as standard_concept_name,
        m_standard_concept.concept_code as standard_concept_code,
        m_standard_concept.vocabulary_id as standard_vocabulary 
    FROM
        ( SELECT
            * 
        FROM
            `measurement` measurement 
        WHERE
            (
                measurement_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `cb_criteria` cr       
                    WHERE
                        concept_id IN (3006698, 3007794, 3020660, 3025809, 3027358)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 1 
                    AND is_selectable = 1)
            )  
            AND (
                measurement.PERSON_ID IN (SELECT
                    distinct person_id  
                FROM
                    `cb_search_person` cb_search_person  
                WHERE
                    cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `cb_search_person` p 
                    WHERE
                        has_whole_genome_variant = 1 ) 
                    AND cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `cb_search_person` p 
                    WHERE
                        has_ehr_data = 1 ) )
            )) measurement 
    LEFT JOIN
        `concept` m_standard_concept 
            ON measurement.measurement_concept_id = m_standard_concept.concept_id", sep="")

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
measurement_79586544_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "measurement_79586544",
  "measurement_79586544_*.csv")
message(str_glue('The data will be written to {measurement_79586544_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), dataset_79586544_measurement_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  measurement_79586544_path,
  destination_format = "CSV")


# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {measurement_79586544_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(standard_concept_name = col_character(), standard_concept_code = col_character(), standard_vocabulary = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
ecg_df_2 <- read_bq_export_from_workspace_bucket(measurement_79586544_path)

library(tidyverse)
library(bigrquery)

# This query represents dataset "ecg" for domain "procedure" and was generated for All of Us Controlled Tier Dataset v7
dataset_79586544_procedure_sql <- paste("
    SELECT
        procedure.person_id,
        procedure.procedure_concept_id,
        p_standard_concept.concept_name as standard_concept_name,
        p_standard_concept.concept_code as standard_concept_code,
        p_standard_concept.vocabulary_id as standard_vocabulary 
    FROM
        ( SELECT
            * 
        FROM
            `procedure_occurrence` procedure 
        WHERE
            (
                procedure_concept_id IN (SELECT
                    DISTINCT c.concept_id 
                FROM
                    `cb_criteria` c 
                JOIN
                    (SELECT
                        CAST(cr.id as string) AS id       
                    FROM
                        `cb_criteria` cr       
                    WHERE
                        concept_id IN (4163951)       
                        AND full_text LIKE '%_rank1]%'      ) a 
                        ON (c.path LIKE CONCAT('%.', a.id, '.%') 
                        OR c.path LIKE CONCAT('%.', a.id) 
                        OR c.path LIKE CONCAT(a.id, '.%') 
                        OR c.path = a.id) 
                WHERE
                    is_standard = 1 
                    AND is_selectable = 1)
            )  
            AND (
                procedure.PERSON_ID IN (SELECT
                    distinct person_id  
                FROM
                    `cb_search_person` cb_search_person  
                WHERE
                    cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `cb_search_person` p 
                    WHERE
                        has_whole_genome_variant = 1 ) 
                    AND cb_search_person.person_id IN (SELECT
                        person_id 
                    FROM
                        `cb_search_person` p 
                    WHERE
                        has_ehr_data = 1 ) )
            )) procedure 
    LEFT JOIN
        `concept` p_standard_concept 
            ON procedure.procedure_concept_id = p_standard_concept.concept_id", sep="")

# Formulate a Cloud Storage destination path for the data exported from BigQuery.
# NOTE: By default data exported multiple times on the same day will overwrite older copies.
#       But data exported on a different days will write to a new location so that historical
#       copies can be kept as the dataset definition is changed.
procedure_79586544_path <- file.path(
  Sys.getenv("WORKSPACE_BUCKET"),
  "bq_exports",
  Sys.getenv("OWNER_EMAIL"),
  strftime(lubridate::now(), "%Y%m%d"),  # Comment out this line if you want the export to always overwrite.
  "procedure_79586544",
  "procedure_79586544_*.csv")
message(str_glue('The data will be written to {procedure_79586544_path}. Use this path when reading ',
                 'the data into your notebooks in the future.'))

# Perform the query and export the dataset to Cloud Storage as CSV files.
# NOTE: You only need to run `bq_table_save` once. After that, you can
#       just read data from the CSVs in Cloud Storage.
bq_table_save(
  bq_dataset_query(Sys.getenv("WORKSPACE_CDR"), dataset_79586544_procedure_sql, billing = Sys.getenv("GOOGLE_PROJECT")),
  procedure_79586544_path,
  destination_format = "CSV")


# Read the data directly from Cloud Storage into memory.
# NOTE: Alternatively you can `gsutil -m cp {procedure_79586544_path}` to copy these files
#       to the Jupyter disk.
read_bq_export_from_workspace_bucket <- function(export_path) {
  col_types <- cols(standard_concept_name = col_character(), standard_concept_code = col_character(), standard_vocabulary = col_character())
  bind_rows(
    map(system2('gsutil', args = c('ls', export_path), stdout = TRUE, stderr = TRUE),
        function(csv) {
          message(str_glue('Loading {csv}.'))
          chunk <- read_csv(pipe(str_glue('gsutil cat {csv}')), col_types = col_types, show_col_types = FALSE)
          if (is.null(col_types)) {
            col_types <- spec(chunk)
          }
          chunk
        }))
}
ecg_df_3 <- read_bq_export_from_workspace_bucket(procedure_79586544_path)

The data will be written to gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/bq_exports/micah_hysong@researchallofus.org/20260423/condition_79586544/condition_79586544_*.csv. Use this path when reading the data into your notebooks in the future.

Loading gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/bq_exports/micah_hysong@researchallofus.org/20260423/condition_79586544/condition_79586544_000000000000.csv.

The data will be written to gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/bq_exports/micah_hysong@researchallofus.org/20260423/measurement_79586544/measurement_79586544_*.csv. Use this path when reading the data into your notebooks in the future.

Loading gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/bq_exports/micah_hysong@researchallofus.org/20260423/measurement_79586544/measurement_79586544_000000000000.csv.

The data will be written to gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/bq_exports/micah_hysong@researchallofus.org/20260423/procedure_79586544/

In [14]:
unique(ecg_df_1$standard_concept_name)
unique(ecg_df_2$standard_concept_name)
unique(ecg_df_3$standard_concept_name)

[1] "ECG: sinus tachycardia"                                                                             
 [2] "Acute ST segment elevation myocardial infarction due to left coronary artery occlusion"             
 [3] "Acute anterior ST segment elevation myocardial infarction"                                          
 [4] "Stress ECG equivocal"                                                                               
 [5] "ECG: sinus bradycardia"                                                                             
 [6] "Wide QRS complex"                                                                                   
 [7] "EKG: complete atrioventricular block"                                                               
 [8] "Nonspecific ST-T abnormality on electrocardiogram"                                                  
 [9] "ST segment depression"                                                                              
[10] "Prolonged QT interval"                                                                              
[11] "Prolonged PR interval"                                                                              
[12] "Shortened PR interval"                                                                              
[13] "EKG: heart block"                                                                                   
[14] "Acute ST segment elevation myocardial infarction due to right coronary artery occlusion"            
[15] "ECG: Q-T interval abnormal"                                                                         
[16] "ECG: sinus arrhythmia"                                                                              
[17] "EKG: left bundle branch block"                                                                      
[18] "Inferior ST segment elevation"                                                                      
[19] "Exercise tolerance test abnormal"                                                                   
[20] "Subsequent ST segment elevation myocardial infarction"                                              
[21] "Left axis deviation"                                                                                
[22] "EKG: T wave abnormal"                                                                               
[23] "EKG: accelerated junctional rhythm"                                                                 
[24] "EKG: subendocardial infarct"                                                                        
[25] "EKG: right bundle branch block"                                                                     
[26] "Acute ST segment elevation myocardial infarction"                                                   
[27] "Left ventricular hypertrophy by electrocardiogram"                                                  
[28] "Electrocardiogram finding"                                                                          
[29] "EKG: ventricular tachycardia"                                                                       
[30] "EKG: atrial flutter"                                                                                
[31] "Inverted T wave"                                                                                    
[32] "EKG myocardial ischemia"                                                                            
[33] "ST segment elevation"                                                                               
[34] "ECG: atrial fibrillation"                                                                           
[35] "EKG: right ventricle hypertrophy"                                                                   
[36] "ECG: bradycardia"                                                                                   
[37] "Acute ST segment elevation myocardial infarction of inferior wall"                                  
[38] "Right axis deviation"               

[1] "P wave axis"  "R wave axis"  "T wave axis"  "Q-T interval" "P-R Interval"

[1] "Bundle of His recording"                        
 [2] "Intra-atrial recording"                         
 [3] "Right ventricular recording"                    
 [4] "24 Hour ECG"                                    
 [5] "ECG (electrocardiogram) rhythm strip monitoring"
 [6] "EKG ST segment monitoring"                      
 [7] "Electrocardiogram with exercise test"           
 [8] "Masters' stress test, two-step"                 
 [9] "Electrocardiographic procedure"                 
[10] "Signal-averaged electrocardiography"            
[11] "Electrocardiogram, rhythm"                      
[12] "12 lead ECG"                                    
[13] "Continuous ECG monitoring"                      
[14] "Electrocardiographic monitoring"                
[15] "Holter extended electrocardiographic recording" 
[16] "Tilt table test"

In [15]:
control_criteria <- unique(c(ecg_df_1$person_id, ecg_df_2$person_id, ecg_df_3$person_id))
print(paste0(length(control_criteria), " people with ecg"))

control_exclusion<-unique(c(number_condition_entries$person_id, case_definition2, exclusion))
length(control_exclusion)
controls<-setdiff(control_criteria, control_exclusion)
print(paste0(length(controls), " controls"))

[1] "50956 people with ecg"


[1] 26921

[1] "40554 controls"


# Make df

In [16]:
df_cases <- data.frame(
  person_id = cases,
  status = 1
)

df_controls <- data.frame(
  person_id = controls,
  status = 0
)

final_df <- rbind(df_cases, df_controls)
nrow(final_df)
n_distinct(final_df$person_id)

[1] 60123

[1] 60123

In [17]:
# This snippet assumes that you run setup first

# This code saves your dataframe into a csv file in a "data" folder in Google Bucket

# Replace df with THE NAME OF YOUR DATAFRAME
my_dataframe <- final_df

# Replace 'test.csv' with THE NAME of the file you're going to store in the bucket (don't delete the quotation marks)
destination_filename <- 'eMERGE_afib_case_control_df.csv'

########################################################################
##
################# DON'T CHANGE FROM HERE ###############################
##
########################################################################

# store the dataframe in current workspace
write_excel_csv(my_dataframe, destination_filename)

# Get the bucket name
my_bucket <- Sys.getenv('WORKSPACE_BUCKET')

# Copy the file from current workspace to the bucket
system(paste0("gsutil cp ./", destination_filename, " ", my_bucket, "/data/"), intern=T)

# Check if file is in the bucket
system(paste0("gsutil ls ", my_bucket, "/data/*.csv"), intern=T)


character(0)

[1] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/All_SDoH_data.csv"                                               
  [2] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/All_SDoH_data_domain_filtered_60.csv"                            
  [3] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/Area_level_disease_statistics.csv"                               
  [4] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/Case_Control_df.csv"                                             
  [5] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/Case_control_demographics_SDOH_SIRE_cohort.csv"                  
  [6] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/Case_control_demographics_SDOH_cohort.csv"                       
  [7] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/Case_control_demographics_SES_SIRE_cohort.csv"                   
  [8] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/Case_control_demographics_SES_cohort.csv"                        
  [9] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/Demographic_and_ancestry_covariates.csv"                         
 [10] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_HS_predictions_Afib.csv"                           
 [11] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_HS_predictions_Asthma.csv"                         
 [12] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_HS_predictions_BreastC.csv"                        
 [13] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_HS_predictions_CHD.csv"                            
 [14] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_HS_predictions_CKD.csv"                            
 [15] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_HS_predictions_HyperC.csv"                         
 [16] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_HS_predictions_ProstateC.csv"                      
 [17] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_HS_predictions_t1d.csv"                            
 [18] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_HS_predictions_t2d.csv"                            
 [19] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_NHB_predictions_Afib.csv"                          
 [20] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_NHB_predictions_Asthma.csv"                        
 [21] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_NHB_predictions_BreastC.csv"                       
 [22] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_NHB_predictions_CHD.csv"                           
 [23] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_NHB_predictions_CKD.csv"                           
 [24] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_NHB_predictions_HyperC.csv"                        
 [25] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_NHB_predictions_ProstateC.csv"                     
 [26] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_NHB_predictions_t1d.csv"                           
 [27] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_NHB_predictions_t2d.csv"                           
 [28] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_NHW_predictions_Afib.csv"                          
 [29] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_NHW_predictions_Asthma.csv"                        
 [30] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_NHW_predictions_BreastC.csv"                       
 [31] "gs://fc-secure-672eeb92-4859-4ed9-9f59-d4349f3534a0/data/EN_stratified_NHW_predictions_CHD.csv"